In [72]:
import os
import certifi
import requests
from dotenv import load_dotenv


from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain import hub
from langchain.tools import tool
import requests

In [73]:
from langchain.agents import create_react_agent, AgentExecutor

In [74]:
import os
from dotenv import load_dotenv

load_dotenv(
    r"C:\Users\ASUS\Agentic_AI\03_LangChain_Single_ Agents\.env"
)

print("Tavily key loaded:", bool(os.getenv("TAVILY_API_KEY")))

Tavily key loaded: True


In [75]:
os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

WEATHERSTACK_API_KEY = os.getenv("WEATHERSTACK_API_KEY")

In [76]:
search_tool = TavilySearchResults(max_results = 2)

In [77]:
#CUSTOM TOOLS


@tool
def get_weather_data(city: str) -> str:
    """
    Fetch current weather information for a city.
    """

    url = (
        f"https://api.weatherstack.com/current?"
        f"access_key={WEATHERSTACK_API_KEY}&query={city}"
    )

    response = requests.get(url)

    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather data for {city}"

    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']}°C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%"
    )

In [78]:
result = search_tool.invoke("What is capital of France")
result

[{'url': 'https://home.adelphi.edu/~ca19535/page%204.html',
  'content': '## Paris facts: the capital of France in history\n\nBefore Paris, the capital of France was Lyon (under the Romans). Paris first became the capital of France in 508 under King Clovis. After centuries with no unique capital of France, Paris retrieved its status of capital of France under King Philippe Auguste, who reigned between 1180 and 1223. You can see remains of the Philippe August Paris walls in the passageway between the Louvre parking and Louvre Museum\n\nParis remained the capital of France until today, with one four year interruption. During German occupation (WW2 , 1940-1944), the capital of France was Vichy.\n\ngo to top\n\nReference: [...] |  |  |  |  |  |  |  |\n ---  ---  --- \n| Home | Spain | Sydney | San Francisco | Paris | Las Vegas | Maui |\n\n  \n Paris, France\n\n  \n\n## Paris facts: Paris, the capital of France\n\nParis is the capital of France, the largest country of Europe with 550 000 km

In [79]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv(
    r"C:\Users\ASUS\Agentic_AI\03_LangChain_Single_ Agents\.env",
    override=True,
)

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY was not loaded from the project .env file")

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    api_key=api_key,
)

In [80]:
result = llm.invoke("What year it is?")
result

AIMessage(content='The current year is 2023.', response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 12, 'total_tokens': 20, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini', 'system_fingerprint': 'fp_9ac1f10bfa', 'finish_reason': 'stop', 'logprobs': None}, id='run-6eb4402e-3dcf-4edf-9a96-7ae8a0f63586-0')

In [81]:
#Prompt 


prompt = hub.pull("hwchase17/react")

#in langchain hub pre-defined prompt are already available, so we can use pre define prompt to create our application

#and for create react_agent this prompt (hwchase17/react) is recomended to use

c:\Users\ASUS\Agentic_AI\.venv312\Lib\site-packages\langchain\hub.py:86: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  res_dict = client.pull_repo(owner_repo_commit)


In [82]:
#hwchase17/react > Prompt> 

# Answer the following questions as best you can. You have access to the following tools:

# {tools}

# Use the following format:

# Question: the input question you must answer
# Thought: you should always think about what to do
# Action: the action to take, should be one of [{tool_names}]
# Action Input: the input to the action
# Observation: the result of the action
# ... (this Thought/Action/Action Input/Observation can repeat N times)
# Thought: I now know the final answer
# Final Answer: the final answer to the original input question

# Begin!

# Question: {input}
# Thought:{agent_scratchpad}


In [83]:
tools = [search_tool, get_weather_data]

In [84]:
# CREATE AGENT 

agent = create_react_agent(
    llm= llm,
    tools = tools,
    prompt = prompt #we are telling our agent how to instruct with tools, how to work, each and every instruction is given inside this prompt
)

In [85]:
#so now we have to run this agent object using AgentExecutor

#EXECUTOR

agent_executor = AgentExecutor(
    agent=agent,
    tools= tools,
    verbose=True #True means what ever agent will plan, execute we will be able to see in terminal
)

In [88]:
#RUN

response = agent_executor.invoke({
    "input": (
        "find the capital of Jharkhand"
        "and then find its current weather"
    )
})

print(response["output"])



> Entering new AgentExecutor chain...
I need to find the capital of Jharkhand first. After that, I will check the current weather for that city. 

Action: tavily_search_results_json  
Action Input: "capital of Jharkhand"  [{'url': 'https://en.wikipedia.org/wiki/Jharkhand', 'content': 'Jharkhand (Hindi: Jhārkhaṇḍ, pronounced ( "Help:IPA/Hindi and Urdu") ⓘ-Flashthomsom-Jharkhand.wav "File:LL-Q1568 (hin)-Flashthomsom-Jharkhand.wav"); lit.\u2009\'the land of forests\') is a landlocked state in eastern India. The state shares borders with the states of West Bengal to the east, Chhattisgarh to the west, Uttar Pradesh to the northwest, Bihar to the north and Odisha to the south. It is the 15th largest state by area, and the 14th largest by population. Hindi is the official language of the state. The city of Ranchi is its capital, and Dumka its sub-capital. The state is known for its waterfalls, hills and holy places; Baidyanath Dham, Parasnath, Dewri and Rajrappa are major religious sites. 